# Kritische Prüfung: hält das, was es verspricht?

Ich prüfe die Implementierung in `heom_full.py` so, wie ein Gutachter es täte, der
annimmt, dass irgendwo geschummelt wurde. Fünf Fragen:

1. Ist es wirklich **ein** durchgängiger Schaltkreis, ohne Auslesen und Neupräparieren?
2. Wird wirklich **kein** Teil der Dynamik klassisch gerechnet?
3. Was genau bleibt klassisch — und was kostet es?
4. **Würde das für eine $10^{30}\times10^{30}$-Dichtematrix mit 100 Qubits funktionieren?**
5. Was würde ein Gutachter sonst noch bemängeln?

Die Antwort auf Frage 4 vorweg, damit niemand sie überliest: **Nein.** Nicht annähernd.
Die Gründe stehen unten, mit Zahlen. Frage 1 bis 3 fallen dagegen positiv aus, und es
ist wichtig, die beiden Befunde nicht zu vermischen: *„der klassische Aufwand ist
unabhängig von der Zahl der Zeitschritte"* ist wahr und wird hier belegt. *„der
klassische Aufwand ist unabhängig von der Systemgröße"* ist falsch und wurde nie
behauptet — aber genau diese Verwechslung ist die Falle.

In [ ]:
import os, sys, time
sys.path.insert(0, os.getcwd())
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

from heom_full import (build_grid_full, build_circuit_full, run_full,
                       qutip_reference, heom_generator, safe_norm2)

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .3})

_C_CM = 2.99792458e10
FS_TO_CM = 1e-15 * 2 * np.pi * _C_CM
KB_CM = 0.6950348004
N = 4
H_FMO = np.array([[12375.0, -87.7,   5.5,  -5.9],
                  [-87.7, 12495.0,  30.8,   8.2],
                  [5.5,      30.8, 12175.0, -53.4],
                  [-5.9,      8.2, -53.4, 12285.0]])
H_FMO = H_FMO - np.trace(H_FMO) / N * np.eye(N)
MODEL = dict(H=H_FMO, lam=3.0, gamma=50.0, T_K=300.0,
             KB_CM=KB_CM, FS_TO_CM=FS_TO_CM)
rho0 = np.diag([1.0, 0, 0, 0]); dt_fs = 10.0; DEPTH, NK = 2, 1
dt_cm = dt_fs * FS_TO_CM

grid = build_grid_full(dt_fs, rho0=rho0, depth=DEPTH, Nk=NK, gauge='ado', **MODEL)

## Prüfung 1 — Ist es wirklich ein durchgängiger Schaltkreis?

Der Vorwurf, den man erheben würde: irgendwo wird zwischendurch ausgelesen, klassisch
nachgerechnet und wieder eingelesen. Die nächste Zelle zählt deshalb ab, was im
Schaltkreis tatsächlich steht — nach Gattertyp.

In [ ]:
qc = build_circuit_full(grid, 8, save_states=True, save_every=4)
ops = qc.count_ops()
print("  Was steht im Schaltkreis (8 Schritte)?")
for name, k in sorted(ops.items(), key=lambda x: -x[1]):
    print(f"    {name:20s} {k:4d}")
print(f"""
  Zu lesen ist das so:
    unitary   {ops.get('unitary', 0):3d}  = 8 Anwendungen DESSELBEN Gatters U
    measure   {ops.get('measure', 0):3d}  = die eine Ancilla, je Schritt einmal
    reset     {ops.get('reset', 0):3d}  = dieselbe Ancilla wieder auf |0>
    save_...  {ops.get('save_statevector', 0):3d}  = SIMULATOR-Anweisung, kein Gatter

  Die Ancilla ist der Preis der Sz.-Nagy-Dilatation: U wirkt auf System+Ancilla, und
  nur der Zweig mit Ancilla = |0> traegt P/s.  Das ist keine Unterbrechung der
  Propagation -- der HEOM-Zustand im 'heom'-Register wird zwischen t=0 und t=T NIE
  gemessen und NIE neu praepariert.

  `save_statevector` ist eine reine Simulator-Anweisung.  Sie liest nichts aus und
  stoert nichts; sie legt eine Kopie beiseite, damit wir Zwischenzeiten auswerten
  koennen, ohne den Lauf n-mal zu wiederholen.  Auf echter Hardware entfaellt sie
  ersatzlos, und man laesft den Schaltkreis je Auslesezeit einmal.""")
assert set(ops) <= {'unitary', 'measure', 'reset', 'save_statevector', 'x'}, ops
print("\n  GEPRUEFT: kein einziges Gatter ausser U, der Ancilla-Messung und ihrem Reset.")

## Prüfung 2 — Wird wirklich kein Teil der Dynamik klassisch gerechnet?

Hier liegt der eigentliche Fortschritt gegenüber der komprimierten Variante, also
verdient er die härteste Prüfung. Die komprimierte Version ruft `arnoldi(Pt, x0, m)` auf,
und Arnoldi wendet $\tilde P$ genau $m$-mal auf einen Vektor an. Bei $m=128$ und einem
Lauf über 100 Zeitschritte hat man damit *mehr* Matrix-Vektor-Produkte klassisch
gerechnet als der Quantenschaltkreis Schritte macht.

Das ist zwar keine Zeitpropagation im physikalischen Sinn — der Krylov-Raum ist eine
Basis, keine Trajektorie — aber es ist $m$-mal „die Dynamik anfassen", und es ist genau
der Punkt, an dem ein Gutachter einhaken würde.

Die nächste Zelle greift auf `heom_full.py` zu und zählt jede Anwendung von $P$ oder $A$
auf einen Vektor, die beim Bauen des Gitters vorkommt.

In [ ]:
import inspect
import heom_full

src = inspect.getsource(heom_full.build_grid_full)
verdaechtig = ['arnoldi', 'matrix_power', 'matmul', '@ x', 'P @ ', 'for t in range']
print("  Suche nach Zeitpropagation in build_grid_full:")
for v in verdaechtig:
    n = src.count(v)
    print(f"    '{v}':{'':<18s}{n} Treffer" + ("" if n == 0 else "   <-- ansehen!"))
print(f"    'arnoldi' im ganzen Modul: {inspect.getsource(heom_full).count('arnoldi')}"
      f" Treffer (nur im Kommentar)")

# Gegenprobe: die komprimierte Variante
sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))
import heom_gauge
src_k = inspect.getsource(heom_gauge.build_grid)
print(f"\n  Zum Vergleich, die komprimierte build_grid:")
print(f"    'arnoldi(': {src_k.count('arnoldi(')} Treffer  <-- m Matvecs mit P~")

print("""
  BEFUND: build_grid_full wendet P kein einziges Mal auf einen Vektor an.  Was
  gerechnet wird, ist expm(A dt) und zwei Matrixwurzeln -- Matrix-FUNKTIONEN, keine
  Propagation.  Der Zustand x~_t existiert klassisch zu keinem Zeitpunkt t > 0.

  Die Diagnostik weiter unten iteriert P zwar klassisch, aber das ist Analyse fuer
  dieses Notebook und geht in den Schaltkreis nicht ein.""")

## Prüfung 3 — Was bleibt klassisch, und was kostet es?

Ehrliche Buchführung. Für jeden Schritt: gemessene Zeit, Komplexität, und die
entscheidende Spalte — hängt er von der Zahl der Zeitschritte ab, oder von der
Hilbertraum-Dimension?

In [ ]:
bt = grid['build_times']
n, np_, D = grid['n'], grid['np_'], grid['D']

A, _ = heom_generator(depth=DEPTH, Nk=NK, scale_ados=True, **MODEL)
nnzA = int(np.sum(np.abs(A) > 1e-14))
nnzP = int(np.sum(np.abs(grid['P']) > 1e-14))

print(f"  {'Schritt':34s} {'gemessen':>10s} {'Komplexitaet':>14s} "
      f"{'~Schritte?':>11s} {'~Dimension?':>12s}")
print("  " + "-" * 86)
tab = [
    ('A aufstellen (duenn besetzt)',  bt['generator'],  'O(n) Eintraege', 'nein', 'ja, mild'),
    ('ADO-Skalierung (diagonal)',     0.0,              'O(n)',           'nein', 'ja, mild'),
    ('P = expm(A dt)',                bt['propagator'], 'O(n^3) DICHT',   'nein', 'JA, kubisch'),
    ('Dilatation (2 Matrixwurzeln)',  bt['dilation'],   'O(n^3) DICHT',   'nein', 'JA, kubisch'),
    ('Anfangszustand |0...0>',        0.0,              'O(1)',           'nein', 'nein'),
    ('Ablesung (diagonal, d^2)',      0.0,              'O(d^2)',         'nein', 'nein'),
]
for name, t, cx, a, b in tab:
    print(f"  {name:34s} {t:9.2f}s {cx:>14s} {a:>11s} {b:>12s}")
print("  " + "-" * 86)
print(f"  {'SUMME':34s} {bt['total']:9.2f}s")

print(f"""
  Die Spalte '~Schritte?' ist ueberall 'nein'.  Das ist die Aussage des Papers, und
  sie haelt: ob der Schaltkreis 10 oder 10 Millionen Schritte macht, aendert an dieser
  Tabelle nichts.

  Die Spalte '~Dimension?' ist zweimal 'JA, kubisch'.  Das ist die Aussage, die NICHT
  gilt, und der Grund steht in einer einzigen Zahl:

    A                 {nnzA:7d} Nichtnullen von {n**2:9d}  = {nnzA/n**2:6.2%}  -> DUENN ({nnzA/n:.1f} pro Zeile)
    P = expm(A dt)    {nnzP:7d} Nichtnullen von {n**2:9d}  = {nnzP/n**2:6.2%}  -> DICHT

  Das Matrixexponential zerstoert die duenne Besetzung.  A haette man strukturiert
  aufschreiben und als Schaltkreis bauen koennen, ohne es je als Matrix hinzulegen --
  expm(A dt) nicht.""")

## Prüfung 4 — Würde das für $10^{30}\times10^{30}$ mit 100 Qubits funktionieren?

**Nein.** Nicht knapp, sondern um Größenordnungen, die keinen Sinn mehr ergeben.

100 Qubits bedeuten $n = 2^{100} \approx 1{,}27\cdot10^{30}$. Der Schaltkreis selbst wäre
kein Problem: ein Gatter, $n$-mal angewandt, eine Ancilla. Das Problem ist, dieses Gatter
überhaupt zu **bekommen**. Die nächste Zelle rechnet die drei kritischen Schritte für
$n=2^{100}$ aus und stellt sie neben physikalische Vergleichsgrößen.

In [ ]:
EXA      = 1e18          # FLOP/s, ein Exascale-Rechner
ALTER    = 4.35e17       # s, Alter des Universums
ATOME    = 1e80          # Atome im beobachtbaren Universum
SPEICHER = 1e23          # Bytes, weltweiter Datenbestand (~100 ZB)

for n_bits, label in ((int(np.log2(np_)), 'unser Fall'), (100, 'die Frage')):
    n_dim = 2.0 ** n_bits          # Zustandsregister; die Ancilla kommt dazu
    flops = n_dim ** 3                       # expm / sqrtm, dicht
    byte  = n_dim ** 2 * 16                  # eine dichte komplexe Matrix
    cnots = 4.0 ** (n_bits + 1) / 4          # beliebiges Unitary, Qubits+Ancilla
    print(f"\n  === {n_bits}+1 Qubits, n = {n_dim:.3g}   ({label}) ===")
    print(f"    expm(A dt), dicht : {flops:9.2e} FLOP")
    print(f"                        = {flops/EXA:.2e} s auf einem Exascale-Rechner")
    if flops / EXA > ALTER:
        print(f"                        = {flops/EXA/ALTER:.1e} x das Alter des Universums")
    print(f"    eine Matrix       : {byte:9.2e} Byte")
    if byte > SPEICHER:
        print(f"                        = {byte/SPEICHER:.1e} x der weltweite Datenbestand")
    if byte > ATOME:
        print(f"                        = {byte/ATOME:.1e} x die Atomzahl des Universums")
    print(f"    U als Gatter      : {cnots:9.2e} CNOTs (beliebiges Unitary)")

print("""

  Damit ist die Frage beantwortet.  Es scheitert nicht an einer Ecke, sondern an drei
  unabhaengigen Stellen gleichzeitig, und jede einzelne davon ist toedlich:

    1. expm(A dt) ist eine dichte O(n^3)-Operation.
    2. Die Dilatation braucht zwei Matrixwurzeln, ebenfalls dicht O(n^3).
    3. Selbst wenn U vom Himmel fiele, braeuchte seine Zerlegung in Elementargatter
       ~4^q CNOTs.  Ein BELIEBIGES 101-Qubit-Unitary ist nicht synthetisierbar.

  Punkt 3 ist der grundsaetzlichste: er gilt fuer JEDE Methode, die ein allgemeines
  Unitary auf 100 Qubits als Matrix uebergeben will.  Das ist keine Schwaeche dieses
  Ansatzes, sondern eine Aussage darueber, dass 'gib dem Quantencomputer die Matrix'
  ueberhaupt kein skalierbares Interface ist.""")

### Was von der Konstruktion trotzdem skaliert

Es wäre unfair, hier aufzuhören, denn die Hälfte der Konstruktion überlebt die Frage
mühelos — und das ist kein Zufall, sondern das Ergebnis der Entscheidung, die
Lyapunov-Eichung durch die **diagonale** ADO-Skalierung zu ersetzen.

| Baustein | skaliert? | warum |
|---|---|---|
| Schaltkreisstruktur (ein Gatter, $n$-mal, eine Ancilla) | **ja** | unabhängig von $n$ |
| ADO-Skalierung | **ja** | diagonal, $\rho_{\mathbf n}/\sqrt{\prod_k n_k!\lvert c_k\rvert^{n_k}}$ eintragsweise |
| Anfangszustand | **ja** | $\rho_0\otimes\delta_{\mathbf n,0}$ ist ein Rechenbasiszustand |
| Ablesung | **ja** | die ersten $d^2$ Amplituden, diagonal reskaliert |
| Post-Selektions-Buchführung | **ja** | ein Zählwerk über die Ancilla |
| Erzeuger $A$ | **im Prinzip** | dünn, $O(1)$ Nichtnullen pro Zeile |
| $\exp(A\Delta t)$ | **nein** | dicht |
| Dilatation | **nein** | dicht |
| $U$ als Matrix laden | **nein** | $4^q$ Gatter |

Hätte ich die Lyapunov-Eichung verwendet, stünden in dieser Tabelle drei weitere
„nein": $W$, $W^{1/2}$ und $W^{-1/2}$ sind allesamt dicht, und der Anfangszustand
$W^{1/2}x_0$ wäre kein Basiszustand mehr, also käme eine State-Preparation über
$2^{100}$ Amplituden dazu. Der Wechsel auf die diagonale Eichung kostet laut Abschnitt 2
in `main.ipynb` nur $p(100)=0{,}873$ statt $0{,}998$ — und rettet dafür die halbe Tabelle.

### Was man stattdessen bräuchte

Der Engpass ist präzise lokalisierbar: **$A$ ist dünn und strukturiert, $\exp(A\Delta t)$
ist dicht.** Alle bekannten skalierbaren Verfahren umgehen deshalb genau diesen Schritt
und arbeiten direkt auf $A$:

$$A \;=\; \underbrace{-\tfrac{i}{\hbar}[H,\,\cdot\,]}_{\text{System}} \;-\; \underbrace{\textstyle\sum_k n_k\gamma_k}_{\text{diagonal im ADO-Index}} \;+\; \underbrace{\textstyle\sum_k \big(\mathcal{A}_k^{+} + \mathcal{A}_k^{-}\big)}_{\text{Leiteroperatoren, dünn}}$$

Jeder dieser Terme ist als Schaltkreis auf dem ADO-Index-Register formulierbar, ohne $A$
je als Matrix hinzulegen. Daraus wird ein **Block-Encoding** von $A/\alpha$ gebaut, und
$\exp(A\Delta t)$ folgt per LCU/Taylor-Reihe, Qubitisierung oder LCHS. Für ein dünn
besetztes $A$ kostet das $O(\mathrm{polylog}\,n)$ Gatter statt $O(n^3)$ Operationen.

Drei Gründe, warum das hier trotzdem nicht steht:

1. **Es ist nicht implementiert.** Der Code hier ist ein Prototyp mit dichter linearer
   Algebra, und ich behaupte nicht, dass die Übersetzung trivial wäre.
2. **Die Subnormierung ist ungelöst.** Jedes Block-Encoding liefert $\exp(A\Delta t)/\alpha$
   mit $\alpha>1$, und die Post-Selektion zahlt $\alpha^{-2t}$ — dasselbe Problem, das
   hier die Eichung löst, nur dass $\alpha$ dort von der Sparsity und der
   $\ell_1$-Norm der Terme kommt und nicht frei wählbar ist. Ob eine diagonale
   Skalierung $\alpha$ ähnlich nahe an 1 drücken kann wie hier $\lVert P\rVert$ auf
   $1{,}0007$, ist eine offene Frage.
3. **$\mathcal{L}$ ist nicht hermitesch.** Hamiltonsimulation ist für hermitesche
   Erzeuger gut verstanden; für Liouvillianer ist die Lage unübersichtlicher, und die
   nicht-normale Transiente ($\lVert P\rVert=10{,}7$ ungeeicht!) ist genau die
   Eigenschaft, die die Standardschranken verdirbt.

## Prüfung 5 — Was würde ein Gutachter sonst noch bemängeln?

Sechs Punkte, von denen ich vier für schwerwiegend halte.

In [ ]:
# --- (a) Kein Quantenvorteil, nicht einmal ansatzweise ---------------------
t0 = time.time()
ref = qutip_reference(np.arange(11) * dt_fs, rho0=rho0, depth=DEPTH, Nk=NK, **MODEL)
t_qutip = time.time() - t0

t0 = time.time()                       # denselben Lauf ueber den Schaltkreis
_, _, _inf = run_full(grid, 10, shots=16, save_every=10, verbose=False)
t_circ = time.time() - t0
t_ges = grid['build_times']['total'] + t_circ
print(f"  (a) qutip HEOMSolver, 10 Schritte : {t_qutip:7.2f} s")
print(f"      Gitter, Aufbau                : {grid['build_times']['total']:7.2f} s")
print(f"      Gitter, Lauf (16 Shots)       : {t_circ:7.2f} s")
print(f"      Gitter, gesamt                : {t_ges:7.2f} s"
      f"   -> Faktor {t_ges/t_qutip:.0f} LANGSAMER")
print("      Bei 11 Qubits ist das eine klassische Simulation einer klassischen\n"
      "      Simulation.  Ein Vorteil kann fruehestens dort entstehen, wo n so gross\n"
      "      ist, dass qutip nicht mehr kann -- und genau dort scheitert der Aufbau.")

# --- (b) Gattersynthese --------------------------------------------------
q = grid['n_qubits']
cnots = 23 / 48 * 4 ** q
print(f"\n  (b) Gattersynthese: U ist ein BELIEBIGES {q}-Qubit-Unitary.")
T2 = 200e-6                              # grosszuegige Kohaerenzzeit, 200 us
t_step = cnots * 1e-6                    # 1 us je CNOT
print(f"      Untere Schranke (Shende-Bullock-Markov): ~{cnots:.2e} CNOTs je U.")
print(f"      Bei 1 us je CNOT: {t_step:.2g} s fuer EINEN Zeitschritt.")
print(f"      Gegen eine Kohaerenzzeit von {T2*1e6:.0f} us ist das Faktor "
      f"{t_step/T2:.0e} zu lang --")
print(f"      schon ein einziger Schritt passt nicht, von {100} Schritten ganz zu")
print(f"      schweigen.  Aer wendet die Matrix direkt an und umgeht das komplett.")
print(f"      Hier verliert 'ein Gatter, oft wiederholt' seine Eleganz: das eine")
print(f"      Gatter ist nicht baubar, und bei 101 Qubits waeren es {23/48*4.0**101:.1e}.")

# --- (c) Post-Selektion ueber lange Zeiten -------------------------------
x = np.zeros(grid['n'], complex); x[0] = 1.0
P, s = grid['P'], grid['s']
pt = []
for t in range(501):
    pt.append((np.linalg.norm(x) / (s ** t)) ** 2)
    x = P @ x
pt = np.array(pt)
print(f"\n  (c) Post-Selektion ueber lange Laeufe (klassisch nachgerechnet):")
for t in (10, 100, 250, 500):
    print(f"      t = {t:4d} ({t*dt_fs:6.0f} fs) : p = {pt[t]:.4f}"
          f"   -> {1/pt[t]:6.1f} Shots je akzeptiertem")
print("      Das faellt, aber nicht exponentiell zusammen -- es ist im Wesentlichen die\n"
      "      physikalische Relaxation des HEOM-Zustands, nicht ein Artefakt.  Bei\n"
      "      staerkerer Kopplung wird es schlechter (siehe Parameterstudie im\n"
      "      Hauptordner).")

fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot(np.arange(501) * dt_fs, pt, color='C0', lw=1.8)
ax.set_xlabel('t [fs]'); ax.set_ylabel(r'$p_{\mathrm{total}}(t)$')
ax.set_ylim(0, 1.02); ax.set_title('Was die Post-Selektion ueber 5 ps uebrig laesst')
fig.tight_layout(); plt.show()

### (d) Die Ablesung ist hier geschummelt — bewusst

`run_full` liest den **Statevector** aus dem Simulator und rechnet daraus $\rho_S$. Auf
echter Hardware gibt es keinen Statevector. Man bekommt Zählraten, und daraus die
$d^2$ Einträge von $\rho_S$ zu rekonstruieren ist ein eigenes Problem — im Hauptordner
ist es für die komprimierte Variante gelöst (`run_grid_counts`,
`run_grid_counts_offdiagonal` mit der ++/RR-Basisrotation), hier ist es **nicht**
implementiert.

Das ist keine prinzipielle Lücke: die Ablesung ist hier sogar *einfacher*, weil
$\operatorname{vec}\rho_S$ die ersten $d^2$ Amplituden sind statt einer Linearkombination
über den ganzen Krylov-Raum. Aber es ist unehrlich, den Statevector-Zugriff als „läuft
auf Hardware" zu verkaufen, und die Zelle unten misst, wie viel Norm überhaupt im
Systemblock sitzt — davon hängt ab, ob die Amplitudenschätzung bezahlbar wäre.

### (e) Mid-Circuit-Reset

Jeder Schritt braucht eine Ancilla-Messung **und** einen Reset. Auf heutiger Hardware
ist Mid-Circuit-Reset die fehleranfälligste Operation überhaupt, und hier braucht man
davon einen pro Zeitschritt. Bei 100 Schritten und 99 % Reset-Treue bliebe
$0{,}99^{100}=0{,}37$.

### (f) Tiefe 2 ist eine physikalische Näherung

Sie hat mit dem Verfahren nichts zu tun, begrenzt aber die erreichbare Genauigkeit. Der
Vergleich Tiefe 2 gegen Tiefe 3 in `main.ipynb` beziffert das: der Abschneidefehler ist
um viele Größenordnungen größer als die Abweichung des Schaltkreises vom Solver. Wer
Tiefe 3 will, zahlt $n=2640$, also 13 Qubits und $4^{13}$ statt $4^{11}$ bei der Synthese.

In [ ]:
x = np.zeros(grid['n'], complex); x[0] = 1.0
print("  (d) Wieviel Norm sitzt im Systemblock (die ersten "
      f"{D} von {grid['n']} Koordinaten)?")
print(f"      {'t':>5s} {'Gewicht':>9s}   {'-> Amplitudenschaetzung':>24s}")
for t in range(0, 101):
    if t in (0, 10, 50, 100):
        w = np.linalg.norm(x[:D]) ** 2 / np.linalg.norm(x) ** 2
        print(f"      {t:5d} {w:9.4f}   {'gutartig' if w > 0.1 else 'teuer':>24s}")
    x = grid['P'] @ x
print("""
      Das Gewicht bleibt bei ~0.92 bis 0.97.  Die hoeheren ADOs tragen also wenig
      Norm, und die Amplituden, die man messen will, sind NICHT exponentiell klein.
      Fuer die Ablesung ist das die gute Nachricht -- sie ist der eine Teil, der auch
      bei 100 Qubits bezahlbar bliebe, WENN man das Gatter haette.""")

---

## Fazit

| Frage | Antwort |
|---|---|
| Ein durchgängiger Schaltkreis? | **Ja.** Nur $U$, Ancilla-Messung, Reset. Der HEOM-Zustand wird zwischen $t=0$ und $t=T$ nie angefasst. |
| Wird Dynamik klassisch gerechnet? | **Nein.** Kein Arnoldi, keine Propagation. $\exp(A\Delta t)$ ist eine Matrixfunktion, kein Zeitschritt. |
| Klassischer Aufwand $\propto$ Zeitschritte? | **Nein**, unabhängig davon. Das ist belegt. |
| Genauigkeit gegen qutip? | $\sim10^{-9}$, weit unter dem physikalischen Abschneidefehler von Tiefe 2. |
| **Funktioniert es für $10^{30}\times10^{30}$?** | **Nein.** Drei unabhängige $O(n^3)$- bzw. $O(4^q)$-Blocker. |
| Quantenvorteil? | **Keiner.** Bei 11 Qubits um Größenordnungen langsamer als qutip. |

### Der eine Satz, auf den es ankommt

Der Ansatz macht den klassischen Aufwand unabhängig von der **Zahl der Zeitschritte** —
das ist real, nachgemessen und der Punkt des Papers. Er macht ihn **nicht** unabhängig
von der **Dimension des Hilbertraums**, und wer die beiden Aussagen verwechselt, liest
aus dieser Arbeit ein Skalierungsversprechen heraus, das sie nicht gibt.

Für ein System mit $10^{30}$ Freiheitsgraden müsste man den Schritt
$A \rightarrow \exp(A\Delta t) \rightarrow U$ vollständig ersetzen: nicht die Matrix
ausrechnen und übergeben, sondern $U$ **aus der Struktur von $A$ heraus** als Schaltkreis
bauen (Block-Encoding, LCU, Qubitisierung). Was von der hier gezeigten Konstruktion dabei
überlebt, ist die Hälfte, die diagonal ist: die ADO-Skalierung, der Anfangszustand, die
Ablesung, die Post-Selektions-Buchführung und die Schaltkreisstruktur selbst. Was nicht
überlebt, ist die dichte lineare Algebra — und die ist heute der ganze Aufbau.